In [7]:
# I have two separate csv files one with mitigation policies and one with adaptation policies.
# Some of them are mitigation and adaptation

import pandas as pd

climate_adaptation_policies = pd.read_csv("../data/raw/adaptation_policies.csv")
climate_mitigation_policies = pd.read_csv("../data/raw/mitigation_policies.csv")

# I want to just get the policies that are only adaptation i.e they don't include anything else in the policy objective column
adaptation_only_policies = climate_adaptation_policies[~climate_adaptation_policies['policy_objective'].str.contains('mitigation', case=False, na=False)]
# I want to just get the policies that are only mitigation i.e they don't include anything else in the policy objective column
mitigation_only_policies = climate_mitigation_policies[~climate_mitigation_policies['policy_objective'].str.contains('adaptation', case=False, na=False)]   
# I want to get the policies that are both adaptation and mitigation i.e they include both in the policy objective column
adaptation_and_mitigation_policies = climate_adaptation_policies[climate_adaptation_policies['policy_objective'].str.contains('mitigation', case=False, na=False)]

# Count the number of policies in each category
num_adaptation_only = len(adaptation_only_policies)
num_mitigation_only = len(mitigation_only_policies)
num_adaptation_and_mitigation = len(adaptation_and_mitigation_policies)
print(f"Number of adaptation only policies: {num_adaptation_only}")
print(f"Number of mitigation only policies: {num_mitigation_only}")
print(f"Number of adaptation and mitigation policies: {num_adaptation_and_mitigation}")


# Combine the three categories into one dataframe
combined_policies = pd.concat([adaptation_only_policies, mitigation_only_policies, adaptation_and_mitigation_policies], ignore_index=True)

# Now I need to clean it up so I just have the info im interested in
combined_policies = combined_policies[['policy_id','country_iso', 'policy_name', 'policy_instrument', 'sector', 'policy_description', 'policy_type','decision_date', 'start_date', 'end_date', 'policy_objective']]


# I want to create a new column called classification that classifies the policies into three categories: adaptation only, mitigation only, and both
def classify_policy(row):
    if 'mitigation' in row['policy_objective'].lower() and 'adaptation' in row['policy_objective'].lower():
        return 'both'
    elif 'mitigation' in row['policy_objective'].lower():
        return 'mitigation only'
    elif 'adaptation' in row['policy_objective'].lower():
        return 'adaptation only'
    else:
        return 'unknown'


combined_policies['classification'] = combined_policies.apply(classify_policy, axis=1)
# Save the combined dataframe to a new csv file
combined_policies.to_csv("../data/preprocessed/cpdb_objectives_master.csv", index=False)    

Number of adaptation only policies: 95
Number of mitigation only policies: 5757
Number of adaptation and mitigation policies: 491


In [8]:
# See if there are any unknown policies
unknown_policies = combined_policies[combined_policies['classification'] == 'unknown']
print(f"Number of unknown policies: {len(unknown_policies)}")   

Number of unknown policies: 0


In [9]:
# Now we can start training

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

# get data from classified dataset
df = pd.read_csv("../data/preprocessed/cpdb_objectives_master.csv")

# Create 'Feature' column
df['text_features'] = (
    df['policy_name'].fillna('') + " " + 
    df['policy_instrument'].fillna('') + " " + 
    df['sector'].fillna('') + " " +
    df['policy_type'].fillna('') + " " +
    df['policy_description'].fillna('')
)


# Define X (input) and y (target)
from sklearn.model_selection import train_test_split



X = df['text_features']
y = df['classification']

# Split into 80% Training and 20% Testing
# stratify=y ensures both sets have the same ratio of classes
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Create the Pipeline
# ngram_range=(1, 2) helps the model see "Carbon Tax" as one concept
model_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', ngram_range=(1, 2))),
    ('clf', LogisticRegression(class_weight='balanced'))
])

# Train
print("Training the model...")
model_pipeline.fit(X_train, y_train)

# Evaluate
y_pred = model_pipeline.predict(X_test)
print("\nModel Performance Report:")
print(classification_report(y_test, y_pred))



Training the model...

Model Performance Report:
                 precision    recall  f1-score   support

adaptation only       0.69      0.58      0.63        19
           both       0.46      0.81      0.59        98
mitigation only       0.98      0.92      0.95      1152

       accuracy                           0.90      1269
      macro avg       0.71      0.77      0.72      1269
   weighted avg       0.93      0.90      0.91      1269



In [10]:
# Prepare the text features for the old dataset

df_old = pd.read_csv("../data/preprocessed/cpdb_objectives_master.csv")  # Load the original dataset

df_old['text_features'] = (
    df['policy_name'].fillna('') + " " + 
    df['policy_instrument'].fillna('') + " " + 
    df['sector'].fillna('') + " " +
    df['policy_type'].fillna('') + " " +
    df['policy_description'].fillna('')
)


# Use the trained model to predict
df_old['ml_prediction'] = model_pipeline.predict(df_old['text_features'])

# Create a "Conflict" column to see where the ML disagrees with original labels
df_old['is_mismatch'] = df_old['classification'] != df_old['ml_prediction']

# Filter to see the mismatches
mismatches = df_old[df_old['is_mismatch'] == True]

print(f"Total policies analyzed: {len(df_old)}")
print(f"Number of mismatches found: {len(mismatches)}")
save_path = "../data/mismatches.csv"
mismatches.to_csv(save_path, index=False)
print(f"Mismatches saved to {save_path}")

# Save entire new dataset with policy name, description, original classification, ML prediction, and mismatch flag
full_save_path = "../data/preprocessed/full_predictions.csv"

df_old.to_csv(full_save_path, index=False)
print(f"Full predictions saved to {full_save_path}")

Total policies analyzed: 6343
Number of mismatches found: 312
Mismatches saved to ../data/mismatches.csv
Full predictions saved to ../data/preprocessed/full_predictions.csv


In [ ]:
import numpy as np

# Get the words (features)
feature_names = model_pipeline.named_steps['tfidf'].get_feature_names_out()
# Get the importance weights
coefs = model_pipeline.named_steps['clf'].coef_[0]

# Sort them to see the top words for Adaptation (highest) and Mitigation (lowest)
top_mitigation = [feature_names[i] for i in np.argsort(coefs)[:10]]
top_adaptation = [feature_names[i] for i in np.argsort(coefs)[-10:]]

print(f"Top Mitigation signals: {top_mitigation}")
print(f"Top Adaptation signals: {top_adaptation}")


Top Mitigation signals: ['renewables', 'carbon', 'energy efficiency', 'energy', 'low', 'low carbon', 'efficiency energy', 'fuel', 'efficiency', 'switch']
Top Adaptation signals: ['national adaptation', 'unknown executive', 'america 2013', 'change adaptation', 'adaptation strategy', 'planning general', 'unknown', 'disaster', 'general unknown', 'adaptation']


# What is going on here?

the model has learnt about adaptation nicely because we had a lot of adaptation only policies. but the mitigation data is very noisy it has picked up strange terms like america 2013. we did not have enough good adaptation only policies.



In [ ]:
# Now I've got another database that has more info on adaptation policies
# Let's see if I can read the xlsm file


try:
    # We specify the engine 'openpyxl' which is best for .xlsx/.xlsm files
    df_adapt = pd.read_excel('../data/raw/ADAPT database.xlsm', sheet_name='_Measures', engine='openpyxl')
    print("Successfully read the 'Measures' sheet!")
    print(df_adapt.head())
except Exception as e:
    print(f"Error reading the file: {e}")

adapt_policies = df_adapt[['ID','Name of the measure', 'Definition', 'Main sector(s) addressed', 'Response type']]
adapt_policies.to_csv("../data/preprocessed/adaptation_policies_detailed.csv", index=False) 


In [21]:
# Create 'Feature' column
df_adapt['text_features'] = (
    df_adapt['Name of the measure'].fillna('') + " " + 
    df_adapt['Definition'].fillna('') + " " + 
    df_adapt['Main sector(s) addressed'].fillna('') + " "    
)

df_adapt['classification'] = (
    df_adapt['Response type'].fillna('')
)

# Now lets change the classification to match our original labels
def map_response_type(response):
    if 'Adaptation' in response and 'Mitigation' in response:
        return 'both'
    elif 'Mitigation' in response:
        return 'mitigation only'
    elif 'Adaptation' in response:
        return 'adaptation only'
    else:
        return 'unknown'    
    
df_adapt['classification'] = df_adapt['classification'].apply(map_response_type)
df_adapt_training = df_adapt[['text_features', 'classification']]

# Save the new dataset for training
df_adapt_training.to_csv("../data/preprocessed/adapt_policies_for_training.csv", index=False)
